In [1]:
!pip install duckdb

In [2]:
import duckdb
con = duckdb.connect()

In [5]:
from google.colab import files
uploaded = files.upload()

Saving adm2019.csv to adm2019 (1).csv
Saving adm2020.csv to adm2020 (1).csv
Saving adm2021.csv to adm2021 (1).csv
Saving ef2019a.csv to ef2019a (2).csv
Saving ef2020a.csv to ef2020a (2).csv
Saving ef2021a.csv to ef2021a (2).csv
Saving f1819_f1a.csv to f1819_f1a (1).csv
Saving f1920_f1a.csv to f1920_f1a (1).csv
Saving f2021_f1a.csv to f2021_f1a (1).csv
Saving f2021_f1a_rv.csv to f2021_f1a_rv (1).csv
Saving gr2019.csv to gr2019 (2).csv
Saving gr2020.csv to gr2020 (2).csv
Saving gr2021.csv to gr2021 (2).csv
Saving hd2019.csv to hd2019 (2).csv
Saving hd2020.csv to hd2020 (2).csv
Saving hd2021.csv to hd2021 (2).csv
Saving s2019_oc.csv to s2019_oc (1).csv
Saving s2020_oc.csv to s2020_oc (1).csv
Saving s2021_oc.csv to s2021_oc (1).csv


In [6]:
import duckdb
con = duckdb.connect()

years = [2019, 2020, 2021]

for y in years:
    con.execute(f"CREATE TABLE inst_characterstics_info_{y} AS SELECT *, {y} AS year FROM read_csv_auto('hd{y}.csv', ignore_errors=true);")
    con.execute(f"CREATE TABLE enrollment_info_{y} AS SELECT *, {y} AS year FROM read_csv_auto('ef{y}a.csv', ignore_errors=true);")
    con.execute(f"CREATE TABLE graduation_info_{y} AS SELECT *, {y} AS year FROM read_csv_auto('gr{y}.csv', ignore_errors=true);")
    con.execute(f"CREATE TABLE admission_info_{y} AS SELECT *, {y} AS year FROM read_csv_auto('adm{y}.csv', ignore_errors=true);")
    con.execute(f"CREATE TABLE staff_info_{y} AS SELECT *, {y} AS year FROM read_csv_auto('s{y}_oc.csv', ignore_errors=true);")

In [7]:
con.execute("SHOW TABLES").fetchall()

[('admission_info_2019',),
 ('admission_info_2020',),
 ('admission_info_2021',),
 ('enrollment_info_2019',),
 ('enrollment_info_2020',),
 ('enrollment_info_2021',),
 ('graduation_info_2019',),
 ('graduation_info_2020',),
 ('graduation_info_2021',),
 ('inst_characterstics_info_2019',),
 ('inst_characterstics_info_2020',),
 ('inst_characterstics_info_2021',),
 ('staff_info_2019',),
 ('staff_info_2020',),
 ('staff_info_2021',)]

In [15]:
# Institutional
con.execute("""
CREATE OR REPLACE TABLE hd_all AS
SELECT unitid, instnm, stabbr, control, year FROM inst_characterstics_info_2019
UNION ALL SELECT unitid, instnm, stabbr, control, year FROM inst_characterstics_info_2020
UNION ALL SELECT unitid, instnm, stabbr, control, year FROM inst_characterstics_info_2021;
""")


In [16]:
# Enrollment
con.execute("""
CREATE OR REPLACE TABLE ef_clean AS
SELECT unitid, year, AVG(enrollment) AS enrollment
FROM ef_all
GROUP BY unitid, year;
""")

In [17]:
# Graduation
con.execute("""
CREATE OR REPLACE TABLE ef_clean AS
SELECT unitid, year, AVG(enrollment) AS enrollment
FROM ef_all
GROUP BY unitid, year;
""")

In [18]:

# Admissions
con.execute("""
CREATE OR REPLACE TABLE adm_clean AS
SELECT unitid, year,
       AVG(applcn) AS applcn,
       AVG(admssn) AS admssn
FROM adm_all
GROUP BY unitid, year;
""")

In [9]:
#staffs
con.execute("""
CREATE OR REPLACE TABLE staff_all AS
SELECT unitid, year, HRTOTLT AS staff_count
FROM staff_info_2019
UNION ALL
SELECT unitid, year, HRTOTLT FROM staff_info_2020
UNION ALL
SELECT unitid, year, HRTOTLT FROM staff_info_2021;
""")

In [12]:
#finance

# Create yearly finance tables from uploaded CSVs
con.execute("CREATE TABLE fin_2019 AS SELECT *, 2019 AS year FROM read_csv_auto('f1819_f1a.csv', ignore_errors=true);")
con.execute("CREATE TABLE fin_2020 AS SELECT *, 2020 AS year FROM read_csv_auto('f1920_f1a.csv', ignore_errors=true);")
con.execute("CREATE TABLE fin_2021 AS SELECT *, 2021 AS year FROM read_csv_auto('f2021_f1a.csv', ignore_errors=true);")

con.execute("""
CREATE OR REPLACE TABLE fin_all AS
SELECT unitid, year, F1A01 AS revenue, F1A04 AS expense
FROM fin_2019
UNION ALL
SELECT unitid, year, F1A01, F1A04 FROM fin_2020
UNION ALL
SELECT unitid, year, F1A01, F1A04 FROM fin_2021;
""")

In [14]:
con.execute("""
CREATE OR REPLACE TABLE final_dataset AS
SELECT
    h.instnm,
    h.stabbr,
    h.control,
    h.year,

    e.enrollment,
    g.graduates,
    a.applcn,
    a.admssn,
    f.revenue,
    f.expense,
    s.staff_count,

    -- 🔥 FEATURES
    g.graduates * 1.0 / NULLIF(e.enrollment,0) AS grad_rate,
    a.admssn * 1.0 / NULLIF(a.applcn,0) AS acceptance_rate,
    f.expense * 1.0 / NULLIF(e.enrollment,0) AS spend_per_student,
    e.enrollment * 1.0 / NULLIF(s.staff_count,0) AS student_staff_ratio

FROM hd_all h
LEFT JOIN ef_all e ON h.unitid=e.unitid AND h.year=e.year
LEFT JOIN gr_all g ON h.unitid=g.unitid AND h.year=g.year
LEFT JOIN adm_all a ON h.unitid=a.unitid AND h.year=a.year
LEFT JOIN fin_all f ON h.unitid=f.unitid AND h.year=f.year
LEFT JOIN staff_all s ON h.unitid=s.unitid AND h.year=s.year;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [19]:
con.execute("SELECT * FROM final_dataset LIMIT 5").fetchdf()

,INSTNM,STABBR,CONTROL,year,enrollment,graduates,APPLCN,ADMSSN,revenue,expense,staff_count,grad_rate,acceptance_rate,spend_per_student,student_staff_ratio
0,Alabama A & M University,AL,1,2019,6172,259,9579,8789,104914959,26058147,91,0.041964,0.917528,4221.994005,67.824176
1,Alabama A & M University,AL,1,2019,5273,259,9579,8789,104914959,26058147,91,0.049118,0.917528,4941.806751,57.945055
2,Alabama A & M University,AL,1,2019,5271,259,9579,8789,104914959,26058147,91,0.049137,0.917528,4943.681844,57.923077
3,Alabama A & M University,AL,1,2019,1694,259,9579,8789,104914959,26058147,91,0.152893,0.917528,15382.613341,18.615385
4,Alabama A & M University,AL,1,2019,3577,259,9579,8789,104914959,26058147,91,0.072407,0.917528,7284.916690,39.307692


In [64]:
#enrollment rate(covid trend)

con.execute("""
SELECT year, SUM(enrollment) AS total_enrollment
FROM final_dataset
GROUP BY year
ORDER BY year;
""").fetchdf()

,year,total_enrollment
0,2019,2.483062e+09
1,2020,2.082607e+09
2,2021,2.360663e+09


In [65]:
#graduation trend
con.execute("""
SELECT year, AVG(grad_rate) AS avg_grad_rate
FROM final_dataset
GROUP BY year;
""").fetchdf()

,year,avg_grad_rate
0,2019,4.230910
1,2020,4.703070
2,2021,4.358098


In [69]:
#acceptance rate vs graduation rate for each clg

con.execute("""
SELECT
    instnm,
    acceptance_rate,
    grad_rate
FROM final_dataset
WHERE acceptance_rate IS NOT NULL
""").fetchdf()

,INSTNM,acceptance_rate,grad_rate
0,Alabama A & M University,0.917528,0.041964
1,Alabama A & M University,0.917528,0.049118
2,Alabama A & M University,0.917528,0.049137
3,Alabama A & M University,0.917528,0.152893
4,Alabama A & M University,0.917528,0.072407
...,...,...,...
1904761,"""Montana Bible College""",0.785714,NaN
1904762,"""Montana Bible College""",0.785714,NaN
1904763,"""Montana Bible College""",0.785714,NaN
1904764,"""Montana Bible College""",0.785714,NaN


In [76]:
#avenue rate vs graduate rate

con.execute("""
SELECT
    year,
    AVG(revenue) AS avg_revenue,
    AVG(grad_rate) AS avg_grad_rate
FROM final_dataset
GROUP BY year
ORDER BY year;
""").fetchdf()

,year,avg_revenue,avg_grad_rate
0,2019,1.057802e+08,4.230910
1,2020,1.092140e+08,4.703070
2,2021,1.297776e+08,4.358098


In [77]:
#top instituitions based on graduation rate

con.execute("""
SELECT instnm, grad_rate
FROM final_dataset
WHERE enrollment > 2000
ORDER BY grad_rate DESC
LIMIT 10;
""").fetchdf()

,INSTNM,grad_rate
0,Pennsylvania State University-Main Campus,8.052397
1,"""The Pennsylvania State University""",8.047179
2,"""The Pennsylvania State University""",7.849222
3,"""The Pennsylvania State University""",7.844844
4,Pennsylvania State University-Main Campus,7.722689
5,Pennsylvania State University-Main Campus,7.717746
6,Ultimate Medical Academy-Clearwater,5.904326
7,Ultimate Medical Academy-Clearwater,5.904326
8,The Pennsylvania State University,5.902448
9,Ultimate Medical Academy-Clearwater,5.899212


In [20]:
#how spend per student and student staff ratio affects the graduation rate

con.execute("""
SELECT
    instnm,
    ROUND(spend_per_student,2) AS spend_per_student,
    ROUND(student_staff_ratio,2) AS student_staff_ratio,
    ROUND(grad_rate,2) AS grad_rate
FROM final_dataset
WHERE spend_per_student IS NOT NULL
AND student_staff_ratio IS NOT NULL
ORDER BY grad_rate DESC
LIMIT 10;
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,INSTNM,spend_per_student,student_staff_ratio,grad_rate
0,California Polytechnic State University-San Lu...,194838588.0,1.00,4928.0
1,California Polytechnic State University-San Lu...,194838588.0,1.00,4928.0
2,California Polytechnic State University-San Lu...,194838588.0,1.00,4928.0
3,California Polytechnic State University-San Lu...,194838588.0,0.50,4928.0
4,California Polytechnic State University-San Lu...,194838588.0,0.00,4928.0
5,California Polytechnic State University-San Lu...,194838588.0,0.00,4928.0
6,California Polytechnic State University-San Lu...,194838588.0,0.00,4928.0
7,California Polytechnic State University-San Lu...,194838588.0,0.03,4928.0
8,California Polytechnic State University-San Lu...,194838588.0,0.01,4928.0
9,California Polytechnic State University-San Lu...,194838588.0,0.01,4928.0


In [22]:
#Cost per graduate

con.execute("""
SELECT
    instnm,
    ROUND(expense / NULLIF(graduates,0), 2) AS cost_per_graduate
FROM final_dataset
WHERE expense IS NOT NULL
AND graduates IS NOT NULL
ORDER BY cost_per_graduate DESC
LIMIT 10;
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,INSTNM,cost_per_graduate
0,University of Michigan-Ann Arbor,1.779330e+10
1,University of Michigan-Ann Arbor,1.779330e+10
2,University of Michigan-Ann Arbor,1.779330e+10
3,University of Michigan-Ann Arbor,1.779330e+10
4,University of Michigan-Ann Arbor,1.779330e+10
5,University of Michigan-Ann Arbor,1.779330e+10
6,University of Michigan-Ann Arbor,1.779330e+10
7,University of Michigan-Ann Arbor,1.779330e+10
8,University of Michigan-Ann Arbor,1.779330e+10
9,University of Michigan-Ann Arbor,1.779330e+10
